In [ ]:
# na fainetai pio katharai grammi hub customer --> link order 

# Databricks Data Engineering Bootcamp
## Peripheral Module Β: Data Vault 2.0 Enterprise Modeling

**Context:** The Global Electronics core data platform team requires an agile, historically trackable ecosystem. To eliminate tightly-coupled constraints, you will decouple operational sources into a compliant Data Vault 2.0 Blueprint.

**Task:** You will design and deploy an enterprise Data Vault schema consisting of Business Hubs, Context Satellites, Core Structural Links, and an operational Link Satellite component.

---

## Chapter 1: The Core Architecture Mapping (Theoretical Context)

Before writing data engineering jobs, we map out how operational entities are decoupled into Data Vault primitives to facilitate independent ingestion cycles:

1. **Hub Components (`hub_customer`, `hub_product`):** Capture the immutable Core Business Keys (BKs). They isolate corporate identity independently of source system fluctuations.
2. **Satellite Components (`sat_customer_details`, `sat_product_details`):** Hold the mutable, descriptive contextual attributes (Socio-demographics, pricing states) and store full historicity via system timestamps.
3. **Link Component (`link_order`):** Represents the distinct structural relationships or transactions binding multiple Hub keys together.
4. **Link Satellite Component (`sat_link_order_details`):** Extracted descriptive parameters tightly coupled to the occurrence of a transaction relationship (e.g., non-key transactional dimensions like order dates and line item quantities).

![Data Vault Enterprise Architecture](data_vault_architecture.png)

---

## Chapter 2: Schema Processing Pipeline Execution

### Exercise 2.1: Initializing Staging Data Sources
Read the current operational source data from the internal platform catalog to feed the Data Vault initialization pipeline.

In [ ]:
# Read source references
raw_cust_df   = spark.table("ai_lab.default.customers")
raw_orders_df = spark.table("ai_lab.default.orders")
raw_prod_df   = spark.table("ai_lab.default.products")

### Exercise 2.2: Deploying Business Hubs (Identity Tables)
Generate deterministic SHA-256 hash keys to create separate identity structures for Customer and Product business entities.

In [ ]:
from pyspark.sql import functions as F

# 1. CUSTOMER HUB
hub_customers_df = raw_cust_df.withColumn(
    "customer_hash_key", F.sha2(F.upper(F.trim(F.col("customer_id").cast("string"))), 256)
).select(
    "customer_hash_key", 
    "customer_id", 
    F.current_timestamp().alias("load_datetime"), 
    F.lit("CRM").alias("record_source")
).distinct()

hub_customers_df.write.format("delta").mode("overwrite").saveAsTable("hub_customer")
display(spark.table("hub_customer"))

In [ ]:
# 2. PRODUCT HUB 
# TODO: Use the SHA-256 function to hash the unique product business natural key
hub_products_df = raw_prod_df.withColumn(
    "product_hash_key", F.___(F.upper(F.trim(F.col("___").cast("string"))), 256)
).select(
    "product_hash_key", 
    "product_id", 
    F.current_timestamp().alias("load_datetime"), 
    F.lit("ERP").alias("record_source")
).distinct()

hub_products_df.write.format("delta").mode("overwrite").saveAsTable("hub_product")
display("hub_product")

### Exercise 2.3: Deploying Descriptive Context Satellites
Build the descriptive context table structures. Satellites map changes over time back to their respective parent Hash Key identifiers.

In [ ]:
# PRODUCT SATELLITE 
# TODO: Match the correct parent hash key column name to bind contextual states to the Hub registry
sat_products_df = raw_prod_df.withColumn(
    "product_hash_key", F.sha2(F.upper(F.trim(F.col("product_id").cast("string"))), 256)
).select(
    "___", 
    "product_name", 
    "category", 
    "price", 
    F.current_timestamp().alias("load_datetime"), 
    F.lit("ERP").alias("record_source")
)

sat_products_df.write.format("delta").mode("overwrite").saveAsTable("sat_product_details")
display("sat_product_details")

In [ ]:
# CUSTOMER SATELLITE 
# TODO: Match the correct parent hash key column name to bind contextual states to the Hub registry
sat_customer_df = raw_cust_df.withColumn(
    "customer_hash_key", F.sha2(F.upper(F.trim(F.col("customer_id").cast("string"))), 256)
).select(
    "___", 
    "customer_name", 
    "city", 
    F.current_timestamp().alias("load_datetime"), 
    F.lit("ERP").alias("record_source")
)

sat_products_df.write.format("delta").mode("overwrite").saveAsTable("sat_customer_details")
display("sat_customer_details")

### Exercise 2.4: Deploying Structural Transaction Links
Flatten multi-entity relationships by building a clean connection entity bridging the Customer Hub and Product Hub targets.

In [ ]:
# To successfully map relationships, we flatten nested transactional items first
raw_orders_exploded = raw_orders_df.withColumn("item", F.explode("items")) \
    .withColumn("product_id", F.col("item.product_id")) \
    .withColumn("quantity", F.col("item.quantity"))

# ORDER LINK 
# TODO: Complete the concatenation delimiter token ("||") inside the multi-key structural hash expression
link_orders_df = raw_orders_exploded.withColumn(
    "link_order_hash_key",
    F.sha2(F.concat_ws("___", 
        F.upper(F.trim(F.coalesce(F.col("order_id").cast("string"), F.lit("")))),
        F.upper(F.trim(F.coalesce(F.col("customer_id").cast("string"), F.lit("")))),
        F.upper(F.trim(F.coalesce(F.col("product_id").cast("string"), F.lit(""))))
    ), 256)
).withColumn(
    "customer_hash_key", F.sha2(F.upper(F.trim(F.col("customer_id").cast("string"))), 256)
).withColumn(
    "product_hash_key", F.sha2(F.upper(F.trim(F.col("product_id").cast("string"))), 256)
).select(
    "link_order_hash_key", 
    "order_id", 
    "customer_hash_key", 
    "product_hash_key", 
    F.current_timestamp().alias("load_datetime"), 
    F.lit("WEB_SHOP").alias("record_source")
).distinct()

link_orders_df.write.format("delta").mode("overwrite").saveAsTable("link_order")
display("link_order")

### Exercise 2.5: Deploying a Satellite on top of a Link
Because transactional relationships hold descriptive indicators (order dates, volume units), you must attach an operational Satellite directly to the transaction Link.

In [ ]:
# SATELLITE ON LINK
# TODO: Compute the matching composite relation hash key to append non-key fields safely to the transaction record
sat_link_orders_df = raw_orders_exploded.withColumn(
    "link_order_hash_key",
    F.sha2(F.concat_ws("||", 
        F.upper(F.trim(F.coalesce(F.col("order_id").cast("string"), F.lit("")))),
        F.upper(F.trim(F.coalesce(F.col("customer_id").cast("string"), F.lit("")))),
        F.upper(F.trim(F.coalesce(F.col("product_id").cast("string"), F.lit(""))))
    ), 256)
).select(
    "___", 
    "order_date",
    "quantity",
    F.current_timestamp().alias("load_datetime"),
    F.lit("WEB_SHOP").alias("record_source")
)

sat_link_orders_df.write.format("delta").mode("overwrite").saveAsTable("sat_link_order_details")
display("sat_link_order_details")

print("Data Vault Architecture Status: Enterprise Layout Fully Sampled and Generated.")

In [ ]:
# homework: scd2 type here 